In [1]:
import xarray as xr
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import os, re
import pandas as pd
os.chdir(r'J:\CMIP6_r1i1p1f1')

In [2]:
basin_gdf = gpd.read_file(r'F:\geodata\river_runoff_obs\Tarim.shp')

In [3]:
nc_list = [f[:-3] for f in os.listdir() if f.endswith('.nc')]
csv_list = [f[:-4] for f in os.listdir('zonal_mean_df') if f.endswith('.csv')]
file_list =  [item for item in nc_list if item not in csv_list]

In [4]:
len(file_list)

0

In [ ]:
for file_name in file_list:
    print(file_name)
    variable = file_name.partition('_')[0]
    
    model_name = re.search(r'day_(.*?)_ssp', file_name).group(1)
    ds = xr.open_dataset(file_name+'.nc')
    lon_arr = ds.variables['lon'].values
    lat_arr = ds.variables['lat'].values
    lon_grid, lat_grid = np.meshgrid(lon_arr, lat_arr)
    
    # Create points and track indices
    points = []
    lon_indices = []
    lat_indices = []
    
    for lat_idx, lat in enumerate(lat_arr):
        for lon_idx, lon in enumerate(lon_arr):
            points.append(Point(lon, lat))  # Create Point object
            lon_indices.append(lon_idx)    # Track longitude index
            lat_indices.append(lat_idx)    # Track latitude index
    
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(geometry=points)
    gdf['lon'] = gdf.geometry.x  # Extract longitude (X-coordinate)
    gdf['lat'] = gdf.geometry.y  # Extract latitude (Y-coordinate)
    gdf['lon_index'] = lon_indices  # Add longitude indices
    gdf['lat_index'] = lat_indices  # Add latitude indices
    
    # Check if CRS is set
    if gdf.crs is None:
        # Set the CRS to WGS84 (latitude and longitude in degrees)
        gdf = gdf.set_crs("EPSG:4326")
    gdf = gdf.to_crs(basin_gdf.crs)
        
    # # Save as shapefile
    # gdf.to_file(model_name + '.shp', driver="ESRI Shapefile")
    # 
    # print("Shapefile has been created successfully.")
    
    merged_gdf = gdf.sjoin(basin_gdf, how='left')
    merged_gdf

    filtered_gdf = merged_gdf.dropna(subset=['abbre'])
    filtered_df = filtered_gdf[[ 'lon', 'lat', 'lon_index', 'lat_index', 'abbre']].copy()
    # Convert your DataFrame to lists for indexing
    lon_indices = filtered_df['lon_index'].tolist()
    lat_indices = filtered_df['lat_index'].tolist()
    # Extract the 'pr' variable from the dataset using the indices
    # Assuming 'lon_index' and 'lat_index' are valid indices for your dataset
    pr_data = ds[variable].isel(lon=lon_indices, lat=lat_indices)
    pr_df = pr_data.to_dataframe().reset_index()
    pr_df_merged = pr_df.merge(filtered_df, on=['lat','lon'],how='left')
    pr_df_merged.drop(['lat_index','lon_index'],axis=1,inplace=True)
    mean_pr_df = pr_df_merged.groupby(by=['time','abbre']).mean()
    # Pivot the DataFrame
    pr_df = mean_pr_df.reset_index().pivot(index='time', columns='abbre', values=variable)
    
    # Optional: Rename columns to make them more descriptive (if needed)
    pr_df.columns.name = None  # Remove the name of the columns index if undesired
    
    # Convert cftime.DatetimeNoLeap to pandas datetime
    pr_df.index = pr_df.index.map(lambda x: x.strftime('%Y-%m-%d') if hasattr(x, 'strftime') else x)
    pr_df.index = pd.to_datetime(pr_df.index)
    
    # Format time index as year-month
    pr_df.index = pr_df.index.to_period('D')
    # Display the resulting DataFrame
    pr_df.to_csv(os.path.join('zonal_mean_df', file_name+'.csv'))


tas_day_AWI-CM-1-1-MR_ssp126_r1i1p1f1_gn_20400101-20401231
